# Composite Models: Multiple Models per Dataset

Real datasets often need more than one physical model. A classic case
([issue #46](https://github.com/ai4se1dk/SANS-fitter/issues/46)): a low-Q
diffuse feature plus a high-Q correlation peak, best described by a **dab**
model and a **peak_lorentz** model fitted *simultaneously* against the same
data — sharing a common background.

This notebook demonstrates:

1. Simulating composite data with sasmodels
2. Combining models with `set_models()` (friendly parameter names)
3. Fitting with the bumps engine
4. Per-component curves (`show_components=True`)
5. Sharing parameters across components (`shared=`)

In [ ]:
import numpy as np
from sasdata.dataloader.data_info import Data1D
from sasmodels.core import load_model
from sasmodels.direct_model import DirectModel

from sans_fitter import SANSFitter

## 1. Simulate low-Q diffuse + high-Q peak data

The combined intensity of a `'+'` mixture is

$$I(q) = \text{scale} \cdot \sum_i \text{scale}_i \cdot I_i(q) + \text{background}$$

In [ ]:
q = np.linspace(0.005, 0.3, 80)
data = Data1D(x=q, y=np.ones_like(q), dy=np.full_like(q, 0.05))
data.qmin, data.qmax = q.min(), q.max()

kernel = load_model('dab+peak_lorentz', dtype='single', platform='dll')
truth = dict(
    scale=1.0, background=0.01,
    A_scale=10.0, A_cor_length=50.0,
    B_scale=5.0, B_peak_pos=0.1, B_peak_hwhm=0.01,
)
y = np.asarray(DirectModel(data, kernel)(**truth))
rng = np.random.default_rng(42)
data.y = y + rng.normal(0, 0.02 * y, size=y.shape)
print(f'Simulated {len(q)} points of dab+peak_lorentz data')

## 2. Combine the models

`set_models()` gives every component parameter a friendly prefix
(`dab_cor_length` instead of sasmodels' canonical `A_cor_length`).

In [ ]:
fitter = SANSFitter()
fitter.set_data(data)
fitter.set_models('dab', 'peak_lorentz')
fitter.get_params()

## 3. Configure and fit

The global `scale` and `background` are shared natively; each component has
its own `<name>_scale`. Composite models require the `bumps` engine.

In [ ]:
fitter.set_param('dab_cor_length', value=40, min=1, max=500, vary=True)
fitter.set_param('dab_scale', value=8, min=0.1, max=100, vary=True)
fitter.set_param('peak_lorentz_peak_pos', value=0.08, min=0.01, max=0.5, vary=True)
fitter.set_param('peak_lorentz_peak_hwhm', value=0.008, min=0.001, max=0.1, vary=True)
fitter.set_param('peak_lorentz_scale', value=4, min=0.1, max=100, vary=True)
fitter.set_param('background', value=0.005, min=0, max=0.1, vary=True)

result = fitter.fit(engine='bumps', method='amoeba')

## 4. Per-component curves

`show_components=True` overlays one dashed curve per component, each drawn as
`scale · part_scale · I_part(q)` (background shown implicitly in the total).

In [ ]:
fitter.plot_results(show_components=True)

## 5. Sharing parameters across components

`shared=[...]` collapses a parameter that exists in ≥ 2 components into a
single unprefixed knob driving all of them — here, two sphere populations
with a common contrast.

In [ ]:
q2 = np.linspace(0.005, 0.3, 80)
data2 = Data1D(x=q2, y=np.ones_like(q2), dy=np.full_like(q2, 0.05))
data2.qmin, data2.qmax = q2.min(), q2.max()
kernel2 = load_model('sphere+sphere', dtype='single', platform='dll')
truth2 = dict(
    scale=0.1, background=0.001,
    A_sld=4.0, A_sld_solvent=6.4, A_radius=20.0, A_scale=1.0,
    B_sld=4.0, B_sld_solvent=6.4, B_radius=200.0, B_scale=1.0,
)
data2.y = np.asarray(DirectModel(data2, kernel2)(**truth2))

fitter2 = SANSFitter()
fitter2.set_data(data2)
fitter2.set_models(small='sphere', large='sphere', shared=['sld', 'sld_solvent'])
fitter2.get_params()

In [ ]:
fitter2.set_param('sld', value=4.0, min=0, max=8, vary=True)
fitter2.set_param('sld_solvent', value=6.4, vary=False)
fitter2.set_param('small_radius', value=15, min=5, max=100, vary=True)
fitter2.set_param('large_radius', value=150, min=50, max=1000, vary=True)
fitter2.set_param('scale', value=0.1, min=0.001, max=1, vary=True)
fitter2.set_param('background', value=0.001, min=0, max=0.1, vary=True)

result2 = fitter2.fit(engine='bumps', method='amoeba')
fitter2.plot_results(show_components=True)

## Further reading

- `docs/usage.md` — "Combining Models (Composite Models)" walkthrough
- `examples/composite_model_example.py` — runnable script version
- Equality links (`link_params` / `unlink_params`) for asymmetric sharing
- Raw sasmodels syntax: `set_model('dab+peak_lorentz')` keeps `A_`/`B_` names